# Phần 1: Câu hỏi Trắc nghiệm

## Load data - Đọc dữ liệu

In [1]:
import pandas as pd
import numpy as np

DATA_DIR  = 'dataset/'

# Master Data
products = pd.read_csv(DATA_DIR + 'products.csv')
customers = pd.read_csv(DATA_DIR + 'customers.csv')
promotions = pd.read_csv(DATA_DIR + 'promotions.csv')
geography = pd.read_csv(DATA_DIR + 'geography.csv')

# Transaction Data
orders = pd.read_csv(DATA_DIR + 'orders.csv')
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')
payments = pd.read_csv(DATA_DIR + 'payments.csv')
shipments = pd.read_csv(DATA_DIR + 'shipments.csv')
returns = pd.read_csv(DATA_DIR + 'returns.csv')
reviews = pd.read_csv(DATA_DIR + 'reviews.csv')

# Analytics Data
sales = pd.read_csv(DATA_DIR + 'sales.csv')
# sales_test = pd.read_csv(DATA_DIR + 'sales_test.csv')

# Operational Data
inventory = pd.read_csv(DATA_DIR + 'inventory.csv')
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv')

print("Data loaded successfully!")

C:\Users\Admin\AppData\Local\Temp\ipykernel_38152\2133482116.py:14: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(DATA_DIR + 'order_items.csv')


Data loaded successfully!


## Câu 1: 
> Trong số các khách hàng có nhiều hơn một đơn hàng, **trung vị số ngày giữa hai lần mua liên tiếp** (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ `orders.csv`)
> 
> *A) 30 ngày*
> 
> *B) 90 ngày*
> 
> *C) 144 ngày*
> 
> *D) 365 ngày*

In [2]:
# Chuyển đổi cột 'order_date' sang định dạng datetime (vì có thể đang ở dạng chuỗi)
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Sắp xếp dữ liệu theo 'customer_id' và 'order_date' (để gom nhóm các đơn hàng của cùng một khách hàng và theo thứ tự thời gian)
orders = orders.sort_values(by = ['customer_id', 'order_date'])

# Lọc customer_id có nhiều hơn 1 đơn hàng (tạo cột 'prev_date' để lưu ngày của đơn hàng trước đó)
orders['prev_date'] = orders.groupby('customer_id')['order_date'].shift(1)

# Tính khoảng cách ngày giữa 2 đơn hàng liên tiếp (cột 'gap' = 'order_date' - 'prev_date')
orders['gap'] = (orders['order_date'] - orders['prev_date']).dt.days

# Loại bỏ các giá trị NaN trong cột 'gap' (các đơn hàng đầu tiên của mỗi khách hàng là NaN vì không có đơn hàng trước đó để so sánh)
gap = orders['gap'].dropna()

# Tính trung vị số ngày giữa 2 đơn hàng liên tiếp
median_gap = gap.median()

print(f"Trung vị số ngày giữa 2 đơn hàng liên tiếp: {median_gap} ngày")

Trung vị số ngày giữa 2 đơn hàng liên tiếp: 144.0 ngày


Vậy đáp án là **(C) 144 ngày**.

## Câu 2:
> Phân khúc sản phẩm (`segment`) nào trong `products.csv` có **tỷ suất lợi nhuận gộp trung bình** cao nhất? $$\dfrac{(\text{price} − \text{cogs})}{\text{price}}$$
>
> *A) Premium*
>
> *B) Performance*
>
> *C) Activewear*
>
> *D) Standard*

In [6]:
# Tính tỷ suất lợi nhuận gộp cho từng sản phẩm (gross_margin = (price - cogs) / price)
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

# Tính trung bình tỷ suất lợi nhuận gộp cho từng phân khúc sản phẩm ('segment')
segment_gross_margin = products.groupby('segment')['gross_margin'].mean()

print("Chi tiết các phân khúc sản phẩm với tỷ suất lợi nhuận gộp trung bình:")
print(segment_gross_margin.sort_values(ascending = False).to_string())

# Tìm phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất
best_segment = segment_gross_margin.idxmax()
best_margin = segment_gross_margin.max()

print(f"Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất: '{best_segment}'")
print(f"Giá trị trung bình của tỷ suất lợi nhuận gộp cho phân khúc này: {best_margin:.2%}")

Chi tiết các phân khúc sản phẩm với tỷ suất lợi nhuận gộp trung bình:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất: 'Standard'
Giá trị trung bình của tỷ suất lợi nhuận gộp cho phân khúc này: 31.34%


Vậy đáp án là **(D) Standard**.

## Câu 3:
> Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục **Streetwear** (join `returns` với `products` theo `product_id`), **lý do trả hàng** nào xuất hiện nhiều nhất?
>
> *A) defective*
>
> *B) wrong_size*
>
> *C) changed_mind*
>
> *D) not_as_described*

In [5]:
# Nối bảng 'returns' với 'products' theo 'product_id' (dùng inner join)
returns_products = pd.merge(returns, products, on = 'product_id', how = 'inner')

# Lọc các bản ghi thuộc danh mục 'Streetwear'
streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']

# Đếm tần suất các lý do trả hàng
reason_counts = streetwear_returns['return_reason'].value_counts()

print("Thống kê chi tiết các lý do trả hàng cho sản phẩm Streetwear:")
print(reason_counts.to_string())

# Tìm lý do trả hàng phổ biến nhất
most_common_reason = reason_counts.idxmax()
count_most_common_reason = reason_counts.max()

print(f"Lý do trả hàng phổ biến nhất cho sản phẩm Streetwear: '{most_common_reason}'")
print(f"Số lượng trả hàng với lý do này: {count_most_common_reason}")

Thống kê chi tiết các lý do trả hàng cho sản phẩm Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Lý do trả hàng phổ biến nhất cho sản phẩm Streetwear: 'wrong_size'
Số lượng trả hàng với lý do này: 7626


Vậy đáp án là **(B) wrong_size**.

## Câu 4:
> Trong `web_traffic.csv`, nguồn truy cập (`traffic_source`) nào có **tỷ lệ thoát trung bình** (`bounce_rate`) **thấp nhất** trên tất cả các ngày xuất hiện nguồn đó trong cột `traffic_source`?

## Câu 5:
> Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là `promo_id` không null) xấp xỉ là bao nhiêu?

## Câu 6:
> Trong `customers.csv`, xét các khách hàng có `age_group` khác null, nhóm tuổi nào có **số đơn hàng trung bình trên mỗi khách hàng** cao nhất? $$\dfrac{\text{tổng số đơn}}{\text{số khách hàng trong nhóm}}$$

## Câu 7: 
> Vùng (`region`) nào trong `geography.csv` tạo ra **tổng doanh thu cao nhất** trong `sales_train.csv`?

## Câu 8:
> Trong các đơn hàng có `order_status = ’cancelled’` trong `orders.csv`, **phương thức thanh toán** nào được sử dụng nhiều nhất?

## Câu 9:

> Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có **tỷ lệ trả hàng cao nhất**, được định nghĩa là số bản ghi trong `returns` chia cho số dòng trong `order_items` (join với `products` theo `product_id`)?

## Câu 10: 
> Trong `payments.csv`, **kế hoạch trả góp** nào có **giá trị thanh toán trung bình trên mỗi đơn hàng** cao nhất?